In [ ]:
import streamlit as st
import PyPDF2
import faiss
import numpy as np
import requests
from sentence_transformers import SentenceTransformer

In [ ]:
%%capture
!pip install streamlit PyPDF2 faiss-cpu numpy requests sentence-transformers

In [ ]:
GROQ_API_KEY = "MY-API-KEY"
GROQ_CHAT_MODEL = "llama-3.3-70b-versatile"

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

def local_embedding(texts):
    return embedder.encode(texts, convert_to_numpy=True).tolist()



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def groq_chat(messages, max_tokens=300):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}
    payload = {"model": GROQ_CHAT_MODEL, "messages": messages, "max_tokens": max_tokens}
    resp = requests.post(url, headers=headers, json=payload)
    return resp.json()

In [ ]:
st.set_page_config(page_title="PDF Chatbot with Groq", layout="wide")
st.title("Q&A with Your Docs")

2026-03-29 06:48:37.322 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:48:37.328 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:48:37.539 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-29 06:48:37.542 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:48:37.545 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [ ]:
# Sidebar
st.sidebar.header("⚙️ Settings")

uploaded_files = st.sidebar.file_uploader("Upload PDF(s)", type=["pdf"], accept_multiple_files=True)

2026-03-29 06:48:55.985 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:48:55.986 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:48:55.987 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:48:55.988 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:48:55.990 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:48:55.990 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:48:55.991 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:48:55.992 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
if "vector_index" not in st.session_state:
    st.session_state.vector_index = None
    st.session_state.text_chunks = []
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

2026-03-29 06:49:17.853 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:49:17.854 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:49:17.856 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:49:17.856 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:49:17.857 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:49:17.858 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:49:17.861 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:49:17.862 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [ ]:
if uploaded_files:
    with st.spinner("📚 Processing your PDF(s)..."):
        all_chunks = []
        for uploaded_file in uploaded_files:
            pdf_reader = PyPDF2.PdfReader(uploaded_file)
            text = ""
            for page in pdf_reader.pages:
                text += page.extract_text() or ""
            # Split into chunks
            chunks = [text[i:i+500] for i in range(0, len(text), 500)]
            all_chunks.extend(chunks)
            # Use local embeddings
        embeddings = local_embedding(all_chunks)
        dim = len(embeddings[0])
        index = faiss.IndexFlatL2(dim)
        index.add(np.array(embeddings).astype("float32"))

        st.session_state.vector_index = index
        st.session_state.text_chunks = all_chunks

    st.sidebar.success(f"✅ {len(all_chunks)} chunks stored in memory.")

In [ ]:
col1, col2 = st.columns([1, 2])

with col1:
    st.subheader("Uploaded PDFs")
    if uploaded_files:
        for f in uploaded_files:
            st.write(f"📄 {f.name}")
    else:
        st.write("No PDF uploaded yet.")

with col2:
    st.subheader("Q&A with Your AI")

    query = st.text_input("Ask a question:")
    if query and st.session_state.vector_index:
        # Embed query
        q_emb = np.array(local_embedding([query])).astype("float32")
        # Search top 3
        D, I = st.session_state.vector_index.search(q_emb, k=3)
        context = "\n".join([st.session_state.text_chunks[i] for i in I[0]])

        messages = [
            {"role": "system", "content": "Use the context to answer the user's question accurately."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"}
        ]

        with st.spinner("🤖 Thinking..."):
            resp = groq_chat(messages)
            try:
                answer = resp['choices'][0]['message']['content']
            except:
                answer = str(resp)

        # Save chat history
        st.session_state.chat_history.append({"question": query, "answer": answer})

    # Display chat history
    if st.session_state.chat_history:
        for chat in reversed(st.session_state.chat_history):
            st.markdown(f"**🧑 You:** {chat['question']}")
            st.markdown(f"**🤖 AI:** {chat['answer']}")


2026-03-29 06:50:32.398 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:50:32.399 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:50:32.401 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:50:32.402 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:50:32.404 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:50:32.405 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:50:32.406 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 06:50:32.407 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar